# Q-Learning  
### *Model-free RL, off-policy method*

#### Load the Tic-Tac-Toe environment.

In [60]:
from tic_tac_toe_env import TicTacToe
import random
import numpy as np


In [61]:
def random_move(game: TicTacToe, letter: str):
    board = game.board
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    move = random.choice(game.available_moves())
    return move

In [62]:
def choose_move(game: TicTacToe, letter: str):
    # board = list(game.get_flat_state())
    board = game.board
    n = game.n
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    def empty():
        return game.available_moves()

    def lines():
        all_lines = []
        for i in range(n):
            all_lines.append([i*n+j for j in range(n)])
        for j in range(n):
            all_lines.append([i*n+j for i in range(n)])
        # diag
        all_lines.append([i*n+i for i in range(n)])
        # off-diag
        all_lines.append([i*n+(n-1-i) for i in range(n)])
        return all_lines

    def can_win(marker):
        winning_moves = []
        for line in lines():
            vals = [board[i] for i in line]
            if vals.count(marker) == n-1 and vals.count('_') == 1:
                winning_moves.append(line[vals.index('_')])
        return winning_moves # return the list of indicies for winning moves

    # if our wins is not empty then play any of those moves to win (here just pick the first)
    our_wins = can_win(player)
    if our_wins:
        return our_wins[0]

    # if the opponent can win then we just block the move. From the lecture we should just pick it randomly
    opp_wins = can_win(opponent)
    if opp_wins:
        return np.random.choice(opp_wins).item()
    
    empty_cells = empty()

    first_empty = empty_cells[0]
    
    # play sequentially in the row first
    current_row = first_empty//n # recall that these are flattened indices
    next_in_row = first_empty + 1 # next cell in the same row
    
    # if its truly on the same row and empty then play it, otherwise it might wrap around and not make snese
    if next_in_row < (current_row + 1) * n and next_in_row in empty_cells:
        return next_in_row
    
    # now, if thats the case, then try the cell below
    cell_below = first_empty + n
    if cell_below < n*n and cell_below in empty_cells: # so if its actually valid (which it should be) and its empty then play it
        return cell_below
    
    # else play randomly
    return np.random.choice(empty_cells).item()

#### Setup the Q-Learning Agent.

In [63]:
import random
import pickle

class QLearningAgent:
    def __init__(self, alpha=0.5, gamma=0.9, epsilon=0.1):
        self.q_table = {}     # (state_tuple, action) → Q
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, game):
        """Epsilon-greedy action selection using Q-values."""
        state = tuple(game.board)
        available = game.available_moves()

        # Exploration
        if random.random() < self.epsilon:
            return random.choice(available)

        # Exploitation
        q_values = [self.get_q(state, a) for a in available]
        max_q = max(q_values)
        best_moves = [a for a, q in zip(available, q_values) if q == max_q]
        return random.choice(best_moves)

    def best_action(self, game):
        """Choose best move with no randomness for eval"""
        state = tuple(game.board)
        available = game.available_moves()

        q_values = [self.get_q(state, a) for a in available]
        max_q = max(q_values)
        best_moves = [a for a, q in zip(available, q_values) if q == max_q]
        return random.choice(best_moves)

    def learn(self, state, action, reward, next_state, next_available, done):
        """q-learning update"""
        old_q = self.get_q(state, action)

        if done:
            target = reward
        else:
            future_q = max(self.get_q(next_state, a) for a in next_available)
            target = reward + self.gamma * future_q

        new_q = old_q + self.alpha * (target - old_q)
        self.q_table[(state, action)] = new_q  # Store the updated Q-value


    # save weights
    def save(self, filename="q_table.pkl"):
        with open(filename, "wb") as f:
            pickle.dump(self.q_table, f)

    def load(self, filename="q_table.pkl"):
        with open(filename, "rb") as f:
            self.q_table = pickle.load(f)


#### Train the Q-Learning Agent

In [64]:
def train(agent_x, agent_o, n, episodes=200000, epsilon_decay=0.9999):
    epsilon = 1.0 

    wins = []
    draws = []
    losses = []

    win_count = 0
    draw_count = 0
    loss_count = 0

    for episode in range(episodes):
        game = TicTacToe(n=n)
        state_x = tuple(game.board)
        state_o = state_x
        letter = 'X'    # X starts

        while True:
            if letter == 'X':
                action = agent_x.choose_action(game)
                game.make_move(action, 'X')

                next_state = tuple(game.board)
                next_available = game.available_moves()

                reward = get_reward(game, 'X')
                done = game.current_winner == 'X' or not game.empty_squares()

                agent_x.learn(state_x, action, reward,
                              next_state, next_available, done)

                if done:
                    break

                state_x = next_state
                letter = 'O'

            else:  # O plays
                action = agent_o.choose_action(game)
                game.make_move(action, 'O')

                next_state = tuple(game.board)
                next_available = game.available_moves()

                reward = get_reward(game, 'O')
                done = game.current_winner == 'O' or not game.empty_squares()

                agent_o.learn(state_o, action, reward,
                              next_state, next_available, done)

                if done:
                    break

                state_o = next_state
                letter = 'X'

        # ---- TRACK OUTCOMES ----
        if game.current_winner == 'X':
            win_count += 1
        elif game.current_winner == 'O':
            loss_count += 1
        else:
            draw_count += 1

        # ---- EVERY 1000 EPISODES STORE RATES ----
        if (episode + 1) % 1000 == 0:
            total = win_count + draw_count + loss_count
            wins.append(win_count / total)
            draws.append(draw_count / total)
            losses.append(loss_count / total)

            # Reset counters for next block
            win_count = 0
            draw_count = 0
            loss_count = 0

        # ---- EPSILON DECAY ----
        epsilon *= epsilon_decay
        agent_x.epsilon = epsilon
        agent_o.epsilon = epsilon

    # Print final results
    print("Win rates per 1000:", wins)
    print("Draw rates per 1000:", draws)
    print("Loss rates per 1000:", losses)

    return wins, draws, losses




In [65]:
def get_reward(game, letter):
    """the rewards for winning, losing, and non-terminal states."""
    if game.current_winner == letter:
        return 1  # Positive reward for winning
    elif game.current_winner == 'O' and letter == 'X':
        return -1  # Negative reward for losing
    elif game.current_winner == 'X' and letter == 'O':
        return -1
    elif not game.empty_squares():
        return 0  # Draw
    else:
        return 0  # Non-terminal state soo neutral reward


In [66]:
#5000000
board_size = 3
num_episodes = 20000

agentX = QLearningAgent()
agentO = QLearningAgent()

train(agentX, agentO, n=board_size, episodes=num_episodes)


Win rates per 1000: [0.588, 0.591, 0.58, 0.557, 0.613, 0.577, 0.564, 0.569, 0.554, 0.578, 0.572, 0.609, 0.611, 0.603, 0.55, 0.623, 0.626, 0.618, 0.698, 0.58]
Draw rates per 1000: [0.13, 0.122, 0.126, 0.136, 0.112, 0.123, 0.13, 0.136, 0.121, 0.122, 0.142, 0.128, 0.111, 0.133, 0.128, 0.116, 0.121, 0.131, 0.096, 0.142]
Loss rates per 1000: [0.282, 0.287, 0.294, 0.307, 0.275, 0.3, 0.306, 0.295, 0.325, 0.3, 0.286, 0.263, 0.278, 0.264, 0.322, 0.261, 0.253, 0.251, 0.206, 0.278]


([0.588,
  0.591,
  0.58,
  0.557,
  0.613,
  0.577,
  0.564,
  0.569,
  0.554,
  0.578,
  0.572,
  0.609,
  0.611,
  0.603,
  0.55,
  0.623,
  0.626,
  0.618,
  0.698,
  0.58],
 [0.13,
  0.122,
  0.126,
  0.136,
  0.112,
  0.123,
  0.13,
  0.136,
  0.121,
  0.122,
  0.142,
  0.128,
  0.111,
  0.133,
  0.128,
  0.116,
  0.121,
  0.131,
  0.096,
  0.142],
 [0.282,
  0.287,
  0.294,
  0.307,
  0.275,
  0.3,
  0.306,
  0.295,
  0.325,
  0.3,
  0.286,
  0.263,
  0.278,
  0.264,
  0.322,
  0.261,
  0.253,
  0.251,
  0.206,
  0.278])

In [67]:
agentX.save("dec6_qlearning_q_values_3x3")

In [51]:
def play_vs_random(agent, agent_letter, turn, n=3, print_game=False):
    game = TicTacToe(n=n)

    if agent_letter == 'X':
        random_letter = 'O'
    else:
        random_letter = 'X'

    # Use deterministic best moves
    agent.epsilon = 0

    while True:
        if print_game:
            print(game)
            print()

        if turn == agent_letter:
            move = agent.best_action(game)
            game.make_move(move, agent_letter)
        else:
            move = random_move(game, random_letter)
            game.make_move(move, random_letter)

        # Check win
        if game.current_winner:
            if turn == agent_letter:
                return 1      # Q-agent wins
            else:
                return -1     # random agent wins

        # Check draw
        if not game.empty_squares():
            return 0

        # Swap turn
        turn = random_letter if turn == agent_letter else agent_letter


In [50]:
def play_ai_first(agent, agent_letter, turn, n=3, print_game=False):
    game = TicTacToe(n=n)

    if agent_letter == 'X':
        random_letter = 'O'
    else:
        random_letter = 'X'

    # Use deterministic best moves
    agent.epsilon = 0

    while True:
        if print_game:
            print(game)
            print()

        if turn == agent_letter:
            move = agent.best_action(game)
            game.make_move(move, agent_letter)
        else:
            move = random_move(game, random_letter)
            game.make_move(move, random_letter)

        # Check win
        if game.current_winner:
            if turn == agent_letter:
                return 1      # Q-agent wins
            else:
                return -1     # random agent wins

        # Check draw
        if not game.empty_squares():
            return 0

        # Swap turn
        turn = random_letter if turn == agent_letter else agent_letter


In [34]:
def evaluate(agent, agent_letter, cycles=15, games_per_cycle=1000, n=board_size):
    win_rates = []
    draw_rates = []
    loss_rates = []

    for cycle in range(cycles):
        wins = draws = losses = 0

        for _ in range(games_per_cycle):
            # agent plays second
            result = play_vs_random(agent, agent_letter, 'X', n=n, print_game=False)
            # agent plays first
            result_ai_first = play_ai_first(agent, agent_letter, 'O', n=n, print_game=False)

            # --- agent second ---
            if result == 1:
                wins += 1
            elif result == 0:
                draws += 1
            else:
                losses += 1

            # --- agent first ---
            if result_ai_first == 1:
                wins += 1
            elif result_ai_first == 0:
                draws += 1
            else:
                losses += 1

        total_games = games_per_cycle * 2

        win_rate = wins / total_games
        draw_rate = draws / total_games
        loss_rate = losses / total_games

        win_rates.append(win_rate)
        draw_rates.append(draw_rate)
        loss_rates.append(loss_rate)

        # print(f"\n--- Checkpoint {cycle+1} ---")
        # print(f"Games this cycle: {total_games}")
        # print(f"Wins:  {wins} ({win_rate:.3f})")
        # print(f"Draws: {draws} ({draw_rate:.3f})")
        # print(f"Losses:{losses} ({loss_rate:.3f})")
    print("win", win_rates)
    print("draw", draw_rates)
    print("loss", loss_rates)
    return win_rates, draw_rates, loss_rates



In [59]:
# agent = QLearningAgent()
# agent.load("q_table.pkl")   # load trained weights
# these are against random agent, win rate is m
#5000000
board_size = 10
num_episodes = 20000

agentX = QLearningAgent()
agentO = QLearningAgent()

train(agentX, agentO, n=board_size, episodes=num_episodes)
agent_letter = 'X'
turn = 'O'
agent = agentX
if agent_letter == 'X':
  agent = agentX
else:
  agent = agentO

print("Board size: ", board_size)
evaluate(agent, agent_letter,  n=board_size)


Win rates per 1000: [0.01, 0.02, 0.01, 0.015, 0.016, 0.017, 0.011, 0.016, 0.011, 0.019, 0.018, 0.013, 0.014, 0.009, 0.016, 0.013, 0.008, 0.009, 0.013, 0.011]
Draw rates per 1000: [0.976, 0.97, 0.978, 0.976, 0.979, 0.972, 0.978, 0.971, 0.974, 0.967, 0.968, 0.977, 0.975, 0.977, 0.969, 0.974, 0.977, 0.981, 0.975, 0.975]
Loss rates per 1000: [0.014, 0.01, 0.012, 0.009, 0.005, 0.011, 0.011, 0.013, 0.015, 0.014, 0.014, 0.01, 0.011, 0.014, 0.015, 0.013, 0.015, 0.01, 0.012, 0.014]
Board size:  10
win [0.01, 0.016, 0.0115, 0.0115, 0.0125, 0.012, 0.01, 0.0135, 0.0165, 0.015, 0.013, 0.012, 0.0125, 0.018, 0.0135]
draw [0.974, 0.9715, 0.977, 0.9765, 0.974, 0.975, 0.978, 0.975, 0.9665, 0.9715, 0.9665, 0.9695, 0.9725, 0.9685, 0.9775]
loss [0.016, 0.0125, 0.0115, 0.012, 0.0135, 0.013, 0.012, 0.0115, 0.017, 0.0135, 0.0205, 0.0185, 0.015, 0.0135, 0.009]


([0.01,
  0.016,
  0.0115,
  0.0115,
  0.0125,
  0.012,
  0.01,
  0.0135,
  0.0165,
  0.015,
  0.013,
  0.012,
  0.0125,
  0.018,
  0.0135],
 [0.974,
  0.9715,
  0.977,
  0.9765,
  0.974,
  0.975,
  0.978,
  0.975,
  0.9665,
  0.9715,
  0.9665,
  0.9695,
  0.9725,
  0.9685,
  0.9775],
 [0.016,
  0.0125,
  0.0115,
  0.012,
  0.0135,
  0.013,
  0.012,
  0.0115,
  0.017,
  0.0135,
  0.0205,
  0.0185,
  0.015,
  0.0135,
  0.009])

## playing against human

In [73]:
def play():
    print("Welcome to Tic Tac Toe! You are O. AI is X.")

    # declare the tic tac toe environment
    game = TicTacToe(n=3)
    
    # assign player letters
    human_letter = 'O'
    ai_letter = 'X'

    game.print_board()

    while game.empty_squares():
        # TURN 1: HUMAN
        move = None
        while move not in game.available_moves():
            try:
                move = int(input("Enter your move (0-b): "))
            except ValueError:
                continue
        game.make_move(move, human_letter)
        game.print_board()

        if game.current_winner:
            print("You win!")
            return

        if not game.empty_squares():
            print("It's a tie!")
            return

        # TURN 2: AI
        # the get_move function is where we run the monte carlo simulation
        ai_move = ai.best_action(game)

        game.make_move(ai_move, ai_letter)
        print(f"AI moves at {ai_move}")
        game.print_board()

        if game.current_winner:
            print("AI wins!")
            return

        if not game.empty_squares():
            print("It's a tie!")
            return

In [71]:
ai = QLearningAgent()
ai.load("dec6_qlearning_q_values_3x3")

In [74]:
play()

Welcome to Tic Tac Toe! You are O. AI is X.
|   |   |   |
|   |   |   |
|   |   |   |
| O |   |   |
|   |   |   |
|   |   |   |
AI moves at 6
| O |   |   |
|   |   |   |
| X |   |   |
| O | O |   |
|   |   |   |
| X |   |   |
AI moves at 2
| O | O | X |
|   |   |   |
| X |   |   |
| O | O | X |
| O |   |   |
| X |   |   |
AI moves at 4
| O | O | X |
| O | X |   |
| X |   |   |
AI wins!
